In [ ]:
import glob
import json
import os

import numpy as np
import pandas as pd
import seaborn as sns
sns.set()

In [ ]:
train_sets = glob.glob("../data/raw/*/*/*train*.jsonl")
dev_sets = glob.glob("../data/raw/*/*/*dev*.jsonl")
test_sets = glob.glob("../data/raw/*/*/*test*.jsonl")

In [ ]:
train = {}
for path in train_sets:
    path_split = path.split("\\")
    label = path_split[1] + "_" + path_split[2]
    if "dta" in path:
        noise_level = path_split[-1].split("l")[1][0]
        label += f"_{noise_level}"
    if "unmatched" in path:
        label += "_unmatched"
    with open(path, encoding="utf8") as f:
        set_train = []
        for l in f.readlines():
            set_train.append(json.loads(l))
    train[label] = set_train

In [ ]:
dev = {}
for path in dev_sets:
    path_split = path.split("\\")
    label = path_split[1] + "_" + path_split[2]
    if "dta" in path:
        noise_level = path_split[-1].split("l")[1][0]
        label += f"_{noise_level}"
    if "unmatched" in path:
        label += "_unmatched"
    with open(path, encoding="utf8") as f:
        set_dev = []
        for l in f.readlines():
            set_dev.append(json.loads(l))
    dev[label] = set_dev

In [ ]:
test = {}
for path in test_sets:
    path_split = path.split("\\")
    label = path_split[1] + "_" + path_split[2]
    if "dta" in path:
        noise_level = path_split[-1].split("l")[1][0]
        label += f"_{noise_level}"
    if "unmatched" in path:
        label += "_unmatched"
    with open(path, encoding="utf8") as f:
        set_test = []
        for l in f.readlines():
            set_test.append(json.loads(l))
    test[label] = set_test

In [ ]:
records = []
for dataset_label, dataset in train.items():
    [records.append((dataset_label, u, "train")) for u in dataset]

for dataset_label, dataset in dev.items():
    [records.append((dataset_label, u, "dev")) for u in dataset]

for dataset_label, dataset in test.items():
    [records.append((dataset_label, u, "test")) for u in dataset]

In [ ]:
unit_df = pd.DataFrame.from_records(records, columns=["dataset_label", "unit", "split"])
unit_df["language"] = unit_df["dataset_label"].apply(lambda x: x.split("_")[1])
unit_df["primary_dataset_name"] = unit_df["dataset_label"].apply(lambda x: x.split("_")[0])
unit_df["transcription_unit_scope"] = unit_df["unit"].apply(lambda x: x["document_metadata"]["transcription_unit_scope"])
unit_df["ground_truth_tokens"] = unit_df["unit"].apply(lambda x: x["ground_truth"]["num_tokens"])
unit_df["ground_truth_quality_score"] = unit_df["unit"].apply(lambda x: x["ground_truth"]["quality_report"].get("ocr_quality_score"))
unit_df["hypothesis_tokens"] = unit_df["unit"].apply(lambda x: x["ocr_hypothesis"]["num_tokens"])
unit_df["hypothesis_quality_score"] = unit_df["unit"].apply(lambda x: x["ocr_hypothesis"]["quality_report"].get("ocr_quality_score"))
unit_df["hypothesis_cer"] = unit_df["unit"].apply(lambda x: x["ocr_hypothesis"]["quality_report"].get("cer"))
unit_df["dta_unmatched"] = unit_df["dataset_label"].apply(lambda x: "unmatched" in x)
unit_df["dta_noise_level"] = unit_df["dataset_label"].apply(lambda x: x.split("_")[-1]).map({"0": "0", "1": "1", "2": "2", "e": "", "r": "", "n": ""})
unit_df.drop(columns=["dataset_label"], inplace=True)
unit_df.info()

### Calculate token counts on various facets

In [ ]:
idx_map = {'icdar2017':0, 'impresso-nzz':1, 'impresso-snippets':2, 'overproof':3, "de":0, "en":1, "fr": 2, "train":0, "dev": 1, "test": 2}

In [ ]:
unit_df.groupby(by=["primary_dataset_name", "language", "split"])[["hypothesis_tokens"]].count().sort_index(level=[0,1,2], key=lambda x: x.map(idx_map)).unstack(level=[2]).astype("Int64").droplevel(level=0, axis=1)[["train", "dev", "test"]]

In [ ]:
unit_df.groupby(by=["primary_dataset_name", "language", "split"])[["hypothesis_tokens"]].sum().sort_index(level=[0,1,2], key=lambda x: x.map(idx_map)).unstack(level=[2]).astype("Int64").droplevel(level=0, axis=1)[["train", "dev", "test"]]

In [ ]:
unit_df.groupby(by=["language"])[["hypothesis_tokens"]].sum()

In [ ]:
lang_hyp_tok_ax = unit_df.groupby(by=["language"])[["hypothesis_tokens"]].sum().plot(kind="bar")
lang_hyp_tok_ax.set_ylabel("Hypothesis Token Count")
lang_hyp_tok_ax.figure.savefig("../reports/figures/data_exploration/tokens_by_lang.png", bbox_inches="tight")

In [ ]:
unit_df.groupby(by=["primary_dataset_name"])[["hypothesis_tokens"]].sum()

In [ ]:
dataset_hyp_tok_ax = unit_df.groupby(by=["primary_dataset_name"])[["hypothesis_tokens"]].sum().plot(kind="bar")
dataset_hyp_tok_ax.set_ylabel("Hypothesis Token Count")
dataset_hyp_tok_ax.figure.savefig("../reports/figures/data_exploration/tokens_by_dataset.png", bbox_inches="tight")

In [ ]:
unit_df.groupby(by=["split"])[["hypothesis_tokens"]].sum().sort_index(key=lambda x:x.map(idx_map))

In [ ]:
split_hyp_tok_ax = unit_df.groupby(by=["split"])[["hypothesis_tokens"]].sum().sort_index(key=lambda x:x.map(idx_map)).plot(kind="bar")
split_hyp_tok_ax.set_ylabel("Hypothesis Token Count")
split_hyp_tok_ax.figure.savefig("../reports/figures/data_exploration/tokens_by_split.png", bbox_inches="tight")

In [ ]:
g = sns.FacetGrid(data=unit_df, row="primary_dataset_name", col="language", sharex=False, sharey=False)
g.map(sns.histplot, "hypothesis_tokens", bins=20, stat="count")
[ax.set_title(ax.get_title().split(" = ")[1].split(" | ")[0] + " | " + ax.get_title().split(" = ")[-1]) for ax in g.axes.flatten()]
g.axes[0,0].figure.suptitle("Translation Unit Token Counts by dataset / language", y=1.02)
g.axes[0,0].figure.savefig("../reports/figures/data_exploration/token_count_distribution_dataset_lang.png", bbox_inches="tight")

### Generate 10k subset per language

In [ ]:
lang_map = (1 / (unit_df.groupby(by=["language"])[["hypothesis_tokens"]].sum() / unit_df.groupby(by=["language"])[["hypothesis_tokens"]].sum().sum())).to_dict()["hypothesis_tokens"]
# lang_map["fr"] = 6
# lang_map["de"] = 0.3
# lang_map["en"] = 1.8

In [ ]:
unit_df["lang_weights"] = unit_df["language"].map(lang_map)

In [ ]:
sample_10k = unit_df.query("split == 'train'").groupby(by=["primary_dataset_name", "language"]).sample(frac=0.06, weights=unit_df.query("split == 'train'")["lang_weights"].values, random_state=486)
sample_10k.groupby(by=["language"])["hypothesis_tokens"].sum()

In [ ]:
sample_10k.columns

In [ ]:
for name, group in sample_10k.groupby(by=["primary_dataset_name", "language"]):
    dataset, lang = name
    if not os.path.exists(f"../data/interim/with_dta/{dataset}/{lang}"):
        os.makedirs(f"../data/interim/with_dta/{dataset}/{lang}")

    if dataset == "dta19":
        matched = group[~group["dta_unmatched"]]
        levels = ["0", "1", "2"]
        for l in levels:
            with open(f"../data/interim/with_dta/{dataset}/{lang}/hipe-ocrepair-bench_v0.9_{dataset}_v1.1_train_{lang}_l{l}_10k_sample.jsonl", "w", encoding="utf8") as f:
                for unit in matched.query("dta_noise_level == @l")["unit"]:
                    json.dump(unit, f)
                    f.write("\n")

    else:
        with open(f"../data/interim/with_dta/{dataset}/{lang}/hipe-ocrepair-bench_v0.9_{dataset}_v1.1_train_{lang}_10k_sample.jsonl", "w", encoding="utf8") as f:
            for unit in group["unit"]:
                json.dump(unit, f)
                f.write("\n")